# Dayflow + RAG (LangChain + Chroma)

기존 **Dayflow** 챗봇(폼 입력 → 단일 프롬프트 → JSON → HTML 카드 구조)에
**RAG(검색 증강 생성)** 를 통합한 버전입니다.

**핵심 설계**
1. 문서를 업로드하지 않으면 → 기존 동작과 100% 동일 (일반 지식 기반 추천)
2. 문서를 업로드하면 → 청킹 → 임베딩 → Chroma 벡터스토어에 저장 → '취향/목적' 텍스트로 검색 → 검색된 내용을 프롬프트에 `[참고 자료]`로 삽입
3. 모델이 참고 자료를 일반 지식보다 우선 반영하도록 시스템 프롬프트에 명시

**범용성**: 문서 종류를 가리지 않습니다 (맛집 리스트, 회사 복지 문서, 동아리 자료 등 어떤 PDF/TXT/CSV든 동일하게 동작).

---

### 실행 순서
1. 아래 셀들을 순서대로 실행하세요 (패키지 설치 → 설정 → 함수 정의 → UI 실행)
2. 첫 실행 시 API key 입력 프롬프트가 뜹니다
3. 마지막 셀 실행 시 Gradio 공유 링크가 생성됩니다 (`share=True`)


In [3]:
# 패키지 설치 (Colab 환경 기준 — 처음 한 번만 실행하면 됩니다)
!pip install openai gradio langchain langchain-openai langchain-community chromadb pypdf -q


## 1. 기본 설정

API 클라이언트와 임베딩 모델을 초기화합니다.

> ⚠️ **참고**: 임베딩(`OpenAIEmbeddings`)이 기존 게이트웨이(`BASE_URL`)에서 `/v1/embeddings`를 지원하지 않으면 에러가 날 수 있습니다. 이 경우 표준 OpenAI API 키를 따로 입력해야 할 수 있습니다 — 실행 후 에러 메시지를 확인해주세요.


In [4]:
import getpass
import json
import os
import shutil
import tempfile

import gradio as gr
from openai import OpenAI

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader

# ── 기본 설정 ────────────────────────────────────────────────
BASE_URL = "https://factchat-cloud.mindlogic.ai/v1/gateway"
API_KEY = getpass.getpass("API key를 입력하세요: ")

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

MODEL = "gpt-5.2"

# 임베딩은 OpenAI 표준 엔드포인트 호환이 필요합니다.
# 게이트웨이가 /v1/embeddings 를 지원하지 않는 경우, 표준 OpenAI API 키를 따로 써야 할 수 있습니다.
embeddings = OpenAIEmbeddings(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="text-embedding-3-small",
)

# 세션 동안 유지되는 임시 Chroma 디렉토리 (Gradio 단일 프로세스 데모 기준)
CHROMA_DIR = tempfile.mkdtemp(prefix="dayflow_chroma_")
vectorstore = None  # 문서가 업로드되기 전까지는 None 유지 → RAG 비활성 상태


API key를 입력하세요: ··········


## 2. 상수 (시스템 메시지 / 웰컴 메시지)

기존 Dayflow와 동일하되, 시스템 메시지에 "참고 자료가 주어지면 우선 반영" 지시를 추가했습니다.


In [5]:
SYSTEM_MESSAGE = """
당신은 AI 라이프스타일 컨시어지 Dayflow입니다.
슬로건: Go with your own flow

당신의 역할은 사용자의 관계 유형, 취향, 분위기, 시간, 예산, 목적을 분석하여
가장 자연스럽고 만족도 높은 오프라인 경험 일정을 설계하는 것입니다.

절대 단순 장소 나열 방식으로 답변하지 않습니다.
반드시 감정 흐름, 이동 연결성, 현실적 비용 배분을 고려하여
하루 경험 전체를 하나의 자연스러운 흐름으로 설계합니다.

반드시 실제 존재하는 구체적인 장소명을 추천해줘.
각 장소의 구글 평점(4.0~5.0 사이)과 대표 메뉴 또는 특징도 함께 제공해줘.

만약 사용자가 업로드한 참고 자료가 함께 주어지면, 그 자료에 등장하는 장소나 정보를
일반 지식보다 우선적으로 활용해서 추천에 반영해줘. 참고 자료에 없는 부분만
너의 일반 지식으로 보완해줘.
"""

WELCOME_MESSAGE = """안녕하세요! 저는 **Dayflow**입니다 \U0001F33F
*Go with your own flow* — 당신의 바이브에 맞는 하루를 설계해드립니다.

소개팅, 친구 행아웃, 가족 저녁, 비즈니스 자리, 혼자만의 하루까지
어떤 모임이든 최적의 코스를 만들어드릴게요.

(선택) 참고하고 싶은 문서가 있다면 아래에 업로드해주세요.
예: 가본 곳 리스트, 회사 회식 장소 후보, 동아리 자료 등 — 무엇이든 가능합니다.

아래 폼을 작성해주시면 바로 시작하겠습니다!"""


## 3. RAG: 문서 적재 & 검색

- `load_document()`: 확장자별로 적절한 LangChain Loader 선택 (PDF / CSV / TXT)
- `ingest_documents()`: 업로드된 파일들을 청킹 → 임베딩 → Chroma에 저장 (Gradio 업로드 콜백에 연결)
- `retrieve_context()`: 벡터스토어가 없으면 빈 문자열 반환(=RAG 비활성, 기존 동작과 동일), 있으면 관련 chunk를 텍스트로 반환


In [6]:
def load_document(file_path):
    """확장자별로 적절한 LangChain Loader 선택."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext == ".csv":
        loader = CSVLoader(file_path, encoding="utf-8")
    else:
        # txt, md 등 일반 텍스트
        loader = TextLoader(file_path, encoding="utf-8")
    return loader.load()


def ingest_documents(files):
    """Gradio File 업로드 결과(파일 경로 리스트)를 받아 벡터스토어를 (재)구축."""
    global vectorstore

    if not files:
        vectorstore = None
        return "업로드된 문서가 없습니다. 일반 지식 기반으로 추천합니다."

    all_docs = []
    failed = []
    for f in files:
        path = f.name if hasattr(f, "name") else f
        try:
            docs = load_document(path)
            # 어떤 파일에서 왔는지 메타데이터에 남겨 출처 표시에 활용
            for d in docs:
                d.metadata["source_file"] = os.path.basename(path)
            all_docs.extend(docs)
        except Exception as e:
            failed.append(f"{os.path.basename(path)} ({e})")

    if not all_docs:
        vectorstore = None
        msg = "문서를 읽는 데 실패했습니다."
        if failed:
            msg += " 오류: " + ", ".join(failed)
        return msg

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(all_docs)

    # 새 문서가 업로드되면 이전 인덱스를 비우고 새로 구축 (데모 단순성을 위함)
    if os.path.exists(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR, ignore_errors=True)
        os.makedirs(CHROMA_DIR, exist_ok=True)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
    )

    status = f"\u2705 {len(files)}개 문서, {len(chunks)}개 청크로 인덱싱 완료. 이제 추천에 참고 자료가 반영됩니다."
    if failed:
        status += f"\n\u26a0\ufe0f 일부 파일 실패: {', '.join(failed)}"
    return status


def retrieve_context(query, k=4):
    """RAG 검색 노드: 벡터스토어가 없으면 빈 문자열(=RAG 비활성), 있으면 관련 chunk를 텍스트로 반환."""
    global vectorstore
    if vectorstore is None:
        return ""

    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    results = retriever.invoke(query)

    if not results:
        return ""

    blocks = []
    for r in results:
        src = r.metadata.get("source_file", "문서")
        blocks.append(f"[출처: {src}]\n{r.page_content}")
    return "\n\n".join(blocks)


## 4. 프롬프트 빌더

기존 Chain-of-Thought(Step 1~6) 구조는 그대로 유지하고, 검색 결과가 있을 때만 `[참고 자료]` 섹션을 프롬프트에 삽입합니다. JSON 스키마에는 `from_reference` 필드를 추가해 참고자료 기반 추천 여부를 표시합니다.


In [7]:
def build_prompt(region, relation, budget, start_time, duration, purpose, vibe, taste, alcohol, transport, context):
    context_section = ""
    if context:
        context_section = f"""
[참고 자료] — 사용자가 업로드한 문서에서 검색된 내용입니다. 일반 지식보다 우선 활용해주세요.
{context}

"""

    return f"""
사용자 정보:
- 지역: {region}
- 관계/모임 유형: {relation}
- 예산: {budget}
- 시작 시간: {start_time}
- 총 이용 시간: {duration}
- 주 목적: {purpose}
- 분위기 스타일: {vibe}
- 취향/관심사: {taste}
- 술 여부: {alcohol}
- 이동 선호: {transport}

{context_section}위 정보를 바탕으로 아래 순서대로 단계별로 추론하여 일정을 설계해줘. (Chain of Thought)

Step 1. 관계 유형과 분위기를 분석하여 전체 감정 흐름 방향을 한 문장으로 정의
Step 2. 초반 장소 선정 — 긴장 완화 또는 자연스러운 시작에 적합한 실제 장소명과 이유
Step 3. 중반 장소 선정 — 편안한 대화와 분위기 유지에 적합한 실제 장소명과 이유
Step 4. 후반 장소 선정 — 분위기 상승 또는 마무리에 적합한 실제 장소명과 이유
Step 5. 각 장소 간 이동 시간과 연결성 검토
Step 6. 코스별 예상 비용 배분 및 총액 계산

추론이 끝나면, 반드시 아래 JSON 형식으로만 최종 결과를 출력해줘.
JSON 외에 다른 텍스트는 절대 포함하지 마. 마크다운 코드블록도 쓰지 마.

{{
  "flow_summary": "전체 감정 흐름 한 문장",
  "course_a": {{
    "title": "코스 A — 안정형",
    "success_rate": "높음",
    "reason": "추천 이유 1~2문장",
    "vibe_flow": "분위기 흐름 한 줄",
    "total_cost": "예: 2만~4만원",
    "total_walk": "예: 도보 합 40분",
    "target": "추천 대상 한 줄",
    "caution": "주의할 점 한 줄",
    "spots": [
      {{
        "time": "15:00\u201316:00",
        "name": "실제 장소명 (예: 카페 어니언 성수점)",
        "type": "장소 유형 (예: 카페 / 레스토랑 / 바)",
        "reason": "추천 이유 1~2문장",
        "menu": "대표 메뉴 또는 특징 (예: 아메리카노 6,000원 / 크루아상)",
        "rating": "구글 평점 (예: \u2b50 4.5)",
        "vibe": "분위기",
        "cost": "예상 비용",
        "move": "다음 장소까지 이동 시간",
        "tip": "추천 포인트",
        "from_reference": "이 장소가 업로드된 참고 자료에서 나왔으면 true, 아니면 false"
      }}
    ]
  }},
  "course_b": {{
    "title": "코스 B — 취향/분위기형",
    "success_rate": "중상",
    "reason": "추천 이유 1~2문장",
    "vibe_flow": "분위기 흐름 한 줄",
    "total_cost": "예: 1만~3만원",
    "total_walk": "예: 도보 합 60분",
    "target": "추천 대상 한 줄",
    "caution": "주의할 점 한 줄",
    "spots": [
      {{
        "time": "15:00\u201316:00",
        "name": "실제 장소명",
        "type": "장소 유형",
        "reason": "추천 이유 1~2문장",
        "menu": "대표 메뉴 또는 특징",
        "rating": "구글 평점 (예: \u2b50 4.3)",
        "vibe": "분위기",
        "cost": "예상 비용",
        "move": "다음 장소까지 이동 시간",
        "tip": "추천 포인트",
        "from_reference": "이 장소가 업로드된 참고 자료에서 나왔으면 true, 아니면 false"
      }}
    ]
  }}
}}
"""


## 5. HTML 렌더러

기존 카드형 UI 렌더링 로직에 "📄 참고자료 기반" 배지만 추가했습니다. 구조나 스타일은 기존과 동일합니다.


In [8]:
def make_map_link(name, region):
    query = (name + " " + region).replace(" ", "+")
    return f"https://www.google.com/maps/search/?api=1&query={query}"


def render_html(data, region, rag_used):
    def course_html(course, color):
        spots_html = ""
        for s in course["spots"]:
            map_url = make_map_link(s.get('name', ''), region)
            ref_badge = ""
            if str(s.get("from_reference", "")).lower() == "true":
                ref_badge = '<span style="background:#eaf1ff;border-radius:6px;padding:3px 9px;color:#2952cc;font-weight:600;">\U0001F4C4 참고자료 기반</span>'

            spots_html += f"""
            <div style="display:flex;gap:14px;padding:16px 0;border-bottom:1px solid #f0f0f0;">
              <div style="min-width:86px;">
                <div style="background:#111;color:#fff;border-radius:8px;padding:5px 8px;font-size:11px;font-weight:700;text-align:center;line-height:1.4;">{s['time']}</div>
              </div>
              <div style="flex:1;">
                <div style="display:flex;align-items:center;gap:8px;margin-bottom:2px;">
                  <div style="font-size:14px;font-weight:800;color:#111;">{s['name']}</div>
                  <div style="font-size:11px;color:#888;background:#f4f4f4;border-radius:5px;padding:2px 7px;">{s['type']}</div>
                </div>
                <div style="font-size:12px;color:#888;margin-bottom:6px;">{s.get('rating','')}</div>
                <div style="font-size:12px;color:#555;margin-bottom:8px;line-height:1.5;">{s['reason']}</div>
                <div style="font-size:12px;color:#666;background:#f9f9f9;border-radius:6px;padding:5px 10px;margin-bottom:8px;">
                  \U0001F37D {s.get('menu','')}
                </div>
                <div style="display:flex;flex-wrap:wrap;gap:6px;font-size:11px;">
                  <span style="background:#f4f4f4;border-radius:6px;padding:3px 9px;color:#333;">\U0001F3AD {s['vibe']}</span>
                  <span style="background:#f4f4f4;border-radius:6px;padding:3px 9px;color:#333;">\U0001F4B0 {s['cost']}</span>
                  <span style="background:#f4f4f4;border-radius:6px;padding:3px 9px;color:#333;">\U0001F6B6 {s['move']}</span>
                  <a href="{map_url}" target="_blank" style="background:#e8f4ea;border-radius:6px;padding:3px 9px;color:#2d7a3a;text-decoration:none;font-weight:600;">\U0001F4CD 지도 보기</a>
                  {ref_badge}
                </div>
                <div style="margin-top:7px;font-size:11px;color:#999;font-style:italic;">\U0001F4A1 {s['tip']}</div>
              </div>
            </div>"""

        return f"""
        <div style="background:#fff;border-radius:16px;border:1.5px solid {color};padding:22px;margin-bottom:20px;">
          <div style="display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:14px;">
            <div>
              <div style="font-size:16px;font-weight:800;color:#111;">{course['title']}</div>
              <div style="font-size:12px;color:#888;margin-top:3px;">{course['vibe_flow']}</div>
            </div>
            <div style="background:{color};color:#fff;border-radius:8px;padding:5px 12px;font-size:11px;font-weight:700;white-space:nowrap;">
              성공률 {course['success_rate']}
            </div>
          </div>
          <div style="background:#f8f8f8;border-radius:8px;padding:10px 14px;font-size:12px;color:#444;margin-bottom:14px;line-height:1.6;">
            {course['reason']}
          </div>
          {spots_html}
          <div style="display:flex;flex-wrap:wrap;gap:10px;margin-top:14px;">
            <div style="flex:1;min-width:120px;background:#f8f8f8;border-radius:10px;padding:10px 14px;">
              <div style="font-size:10px;color:#aaa;margin-bottom:3px;">총 예상 비용</div>
              <div style="font-size:13px;font-weight:700;color:#111;">{course['total_cost']}</div>
            </div>
            <div style="flex:1;min-width:120px;background:#f8f8f8;border-radius:10px;padding:10px 14px;">
              <div style="font-size:10px;color:#aaa;margin-bottom:3px;">총 이동 시간</div>
              <div style="font-size:13px;font-weight:700;color:#111;">{course['total_walk']}</div>
            </div>
            <div style="flex:1;min-width:120px;background:#f8f8f8;border-radius:10px;padding:10px 14px;">
              <div style="font-size:10px;color:#aaa;margin-bottom:3px;">추천 대상</div>
              <div style="font-size:13px;font-weight:700;color:#111;">{course['target']}</div>
            </div>
          </div>
          <div style="margin-top:12px;font-size:11px;color:#b05e00;background:#fff8ee;border-radius:8px;padding:9px 13px;">
            \u26a0\ufe0f {course['caution']}
          </div>
        </div>"""

    rag_badge = ""
    if rag_used:
        rag_badge = """<div style="display:inline-block;margin-top:10px;background:#eaf1ff;color:#2952cc;border-radius:8px;padding:6px 14px;font-size:11px;font-weight:700;">\U0001F4C4 업로드한 참고 자료가 이 추천에 반영되었습니다</div>"""

    return f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:700px;margin:0 auto;padding:16px;">
      <div style="text-align:center;margin-bottom:22px;">
        <div style="font-size:24px;font-weight:900;letter-spacing:-1px;">Dayflow</div>
        <div style="font-size:12px;color:#aaa;margin-top:2px;">Go with your own flow</div>
        <div style="margin-top:14px;background:#f4f4f4;border-radius:10px;padding:12px 16px;font-size:13px;color:#333;line-height:1.6;">
          {data['flow_summary']}
        </div>
        {rag_badge}
      </div>
      {course_html(data['course_a'], '#111')}
      {course_html(data['course_b'], '#888')}
    </div>"""


## 6. 메인 함수

`generate_course()`에서 RAG 검색 노드를 호출합니다. `taste`(취향) + `purpose`(목적) 텍스트로 검색하고, 결과가 있으면 `rag_used=True`로 표시되어 HTML에 배지가 붙습니다.


In [9]:
def generate_course(region, relation, budget, start_time, duration, purpose, vibe, taste, alcohol, transport):
    if not region or not relation:
        return "<p style=\'color:red;padding:16px;\'>지역과 관계/모임 유형은 필수 입력 항목입니다.</p>"

    # RAG 검색: taste(취향)와 purpose(목적)를 합쳐 쿼리로 사용
    rag_query = f"{taste} {purpose}".strip()
    context = retrieve_context(rag_query) if rag_query else ""
    rag_used = bool(context)

    prompt = build_prompt(region, relation, budget, start_time, duration, purpose, vibe, taste, alcohol, transport, context)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt}
        ],
    )
    raw = response.choices[0].message.content

    try:
        raw_clean = raw.strip()
        if raw_clean.startswith("```"):
            raw_clean = raw_clean.split("\n", 1)[1]
        if raw_clean.endswith("```"):
            raw_clean = raw_clean.rsplit("```", 1)[0]
        data = json.loads(raw_clean.strip())
        return render_html(data, region, rag_used)
    except Exception as e:
        return f"<pre style=\'padding:16px;font-size:12px;\'>{raw}</pre>"


## 7. Gradio UI 실행

기존 폼 UI에 문서 업로드 컴포넌트(`gr.File`)와 인덱싱 상태 표시창을 추가했습니다.
실행하면 공유 링크(`share=True`)가 생성됩니다.

**데모 팁**: 문서 업로드 없이 실행하면 기존과 동일하게 동작 → 문서를 올린 후 다시 실행하면 "📄 참고자료 기반" 배지가 붙는 걸 비교해서 보여줄 수 있습니다.


In [10]:
with gr.Blocks(title="Dayflow — Go with your own flow") as demo:

    gr.Markdown("# Dayflow")
    gr.Markdown(WELCOME_MESSAGE)

    gr.Markdown("### (선택) 참고 문서 업로드 — RAG 검색")
    with gr.Row():
        file_upload = gr.File(
            label="문서 업로드 (PDF / TXT / CSV, 여러 개 가능)",
            file_count="multiple",
            file_types=[".pdf", ".txt", ".csv", ".md"],
        )
    ingest_status = gr.Textbox(label="인덱싱 상태", interactive=False, value="아직 업로드된 문서가 없습니다.")
    file_upload.change(fn=ingest_documents, inputs=file_upload, outputs=ingest_status)

    gr.Markdown("---")
    gr.Markdown("### 모임 정보를 입력해주세요")

    with gr.Row():
        region = gr.Textbox(label="지역 *", placeholder="예: 성수, 강남, 연남, 홍대")
        relation = gr.Dropdown(
            label="관계 / 모임 유형 *",
            choices=["소개팅", "썸", "연인", "친구 행아웃", "가족 저녁", "비즈니스", "혼자"],
            value="친구 행아웃"
        )

    with gr.Row():
        budget = gr.Textbox(label="예산", placeholder="예: 1인 5만원, 총 15만원, 가성비 위주")
        start_time = gr.Textbox(label="시작 시간", placeholder="예: 오후 6시")
        duration = gr.Textbox(label="총 이용 시간", placeholder="예: 4시간, 저녁 이후 전부")

    with gr.Row():
        purpose = gr.Textbox(label="주 목적", placeholder="예: 자연스럽게 친해지기, 분위기 좋은 데이트, 특별한 하루")
        vibe = gr.Dropdown(
            label="분위기 스타일",
            choices=["감성적인", "힙한", "조용한", "고급스러운", "자연스러운", "활발한"],
            value="자연스러운"
        )

    with gr.Row():
        taste = gr.Textbox(label="취향 / 관심사", placeholder="예: 전시, 음악, LP바, 사진, 와인, 산책, 야경")
        alcohol = gr.Dropdown(
            label="술 여부",
            choices=["술 X", "가볍게만", "와인 선호", "칵테일 선호", "자유롭게"],
            value="가볍게만"
        )
        transport = gr.Dropdown(
            label="이동 선호",
            choices=["도보 위주", "이동 최소화", "택시 가능"],
            value="도보 위주"
        )

    gr.Markdown("---")
    submit_btn = gr.Button("Dayflow 코스 추천받기", variant="primary")
    output = gr.HTML(label="추천 코스")

    submit_btn.click(
        fn=generate_course,
        inputs=[region, relation, budget, start_time, duration, purpose, vibe, taste, alcohol, transport],
        outputs=output
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6078516c2a7363ade0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6078516c2a7363ade0.gradio.live
